In [2]:
import pandas, pathlib, json

tables_path = pathlib.Path('/mnt/uat_phil_volume_1/uatvm/ntms/tables/')
data_path = pathlib.Path('/mnt/uat_phil_volume_1/uatvm/ntms/data/')
root_path = pathlib.Path('/mnt/uat_phil_volume_1/uatvm/ntms/')

In [126]:
processed_runs = pandas.read_csv(root_path / "UKHSA-NTMs+TB.csv")
for i in ['UNIQUEID', 'SAMPLE_ACCESSION', 'MYCOBACTERIAL_READS', 'UNCLASSIFIED_READS', 'NON_MYCOBACTERIAL_READS', 'PIPELINE_BUILD']:
    processed_runs[i] = None

processed_runs.columns

processed_runs = processed_runs[['RUN_ACCESSION', 'UNIQUEID', 'SAMPLE_ACCESSION',
       'MYCOBACTERIAL_READS', 'UNCLASSIFIED_READS', 'NON_MYCOBACTERIAL_READS',
       'PIPELINE_BUILD', 'bucket_sample_name', 'in_mapping_file', 'has_main_report',
       'has_mutations', 'has_effects', 'has_predictions', 'status',
       'batch_name', 'pipeline_version', 'sample_name', 'remote_sample_name',
       'remote_batch_name', 'remote_batch_id' ]]
processed_runs.set_index('bucket_sample_name', inplace=True)
processed_runs

,RUN_ACCESSION,UNIQUEID,SAMPLE_ACCESSION,MYCOBACTERIAL_READS,UNCLASSIFIED_READS,NON_MYCOBACTERIAL_READS,PIPELINE_BUILD,in_mapping_file,has_main_report,has_mutations,has_effects,has_predictions,status,batch_name,pipeline_version,sample_name,remote_sample_name,remote_batch_name,remote_batch_id
bucket_sample_name,,,,,,,,,,,,,,,,,,,
27b12f67-6a2b-4312-abca-7554c18a4e9e,00142f04-7b41-4c9a-a2e3-66db3c75ffb8,None,None,None,None,None,None,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
495e5dc2-3b57-460c-8354-7dabfb689ebf,001948a5-a8fb-4b51-948d-26b5f3f55bd7,None,None,None,None,None,None,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
a9ca7294-28d1-41aa-af8b-8975ffd545f6,0026862b-e129-436b-9b5f-4d573b705d67,None,None,None,None,None,None,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
a9e4f4d7-d548-4389-b548-35ba0baf66b4,002fcd28-42c5-43ca-8598-c6540d41f133,None,None,None,None,None,None,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
882c5a22-cf23-4709-80d4-7efc111395d9,003eac18-a228-4d12-984e-7cff3ce9f81d,None,None,None,None,None,None,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR6982036,SRR6982036,None,None,None,None,None,None,True,False,False,False,False,NaN,NaN,2.4.1,SRR6982036,90406ae5-4576-46b1-8499-e42840192172,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129
SRR6982033,SRR6982033,None,None,None,None,None,None,True,False,False,False,False,NaN,NaN,2.4.1,SRR6982033,43321ebd-cb71-4f3a-b7e3-84ba43d02b17,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129
SRR6982034,SRR6982034,None,None,None,None,None,None,True,False,False,False,False,NaN,NaN,2.4.1,SRR6982034,cf8b2308-0407-4086-8f7a-6e9780807653,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129


In [127]:
processed_runs.has_mutations.value_counts(dropna=False)

has_mutations
False    52811
Name: count, dtype: int64

In [131]:
len(processed_runs)

52811

In [5]:
json_file = data_path / 'main_report/77434f41-625f-424a-97c5-8d318ce11410_main_report_v2.4.1.json'
with open(json_file) as f:
    data = json.load(f)

In [7]:
data['Assembled NTM Results']

{'Assembled Species': ['Mycobacterium_avium', 'Mycobacterium_intracellulare'],
 'Summary': [{'Name': 'Mycobacterium intracellulare',
   'Num Reads': 1806529,
   'Coverage': 93.0141,
   'Depth': 45.4026},
  {'Name': 'Mycobacterium avium',
   'Num Reads': 217986,
   'Coverage': 90.6699,
   'Depth': 5.93603}],
 'Species': [{'Name': 'Mycobacterium intracellulare',
   'Num Reads': 1806529,
   'Coverage': 93.0141,
   'Mean Depth': 45.4026,
   'Length': 5402402},
  {'Name': 'Mycobacterium avium',
   'Num Reads': 217986,
   'Coverage': 90.6699,
   'Mean Depth': 5.93603,
   'Length': 4956929}],
 'Phylogenic Group': [{'Name': 'Non_tuberculosis_mycobacterium_complex',
   'Coverage': 90.572,
   'Median Depth': 43}],
 'Subspecies': [{'Name': 'Mycobacterium_intracellulare',
   'Coverage': 90.572,
   'Median Depth': 43}],
 'Lineage': [{'Name': 'Mycobacterium_intracellulare_subsp._intracellulare',
   'Coverage': 83.518,
   'Median Depth': 42}]}

In [136]:
# first we need to parse the main_reports and extract the [SPECIES] and populate GENOMES

n_samples = 0

successful_genome = 0
too_few_reads_genome = 0
too_few_reads_id = 0

tables=[]
for i in (data_path / "main_report").rglob("*main_report*.json"):
    if "_mycobacterial_reads" in str(i):
        uid = i.stem.split("_mycobacterial_reads")[0]
    elif ".main_report" in str(i):
        uid = i.stem.split('.main_report')[0]
    else:
        raise ValueError('problem with ', str(i))
    
    # check to see if the UID is in the master table and complain if not
    if not (processed_runs.index==uid).any():
        print(uid, "not in the master table")
        continue
        print('nooo')

    # if it is, record that 
    processed_runs.at[uid, "has_main_report"] = True

    f = open(i)
    data = json.load(f)
    n_samples += 1

    if (
        data["Pipeline Outcome"]
        == "Mycobacterial species identified. Reads mapped to M. tuberculosis (H37Rv v3) too low to proceed to M. tuberculosis complex genome assembly."
    ):
        too_few_reads_genome += 1
        processed_runs.at[uid, "status"] = "too few H37Rv reads"
    elif (
        data["Pipeline Outcome"]
        == "Number of Mycobacterial reads is too low to proceed to Mycobacterial species identification."
    ):
        too_few_reads_id += 1
        processed_runs.at[uid, "status"] = "cannot speciate"
        continue
    elif (
        data["Pipeline Outcome"]
        == "Sufficient reads mapped to M. tuberculosis (H37Rv v3) for genome assembly, resistance prediction and relatedness assessment."
    ):
        processed_runs.at[uid, "status"] = "complete"
        successful_genome += 1
    else:
        print(uid, data["Pipeline Outcome"])
        continue

    # let's build the GENOMES table
    processed_runs.at[uid,'MYCOBACTERIAL_READS'] = data["Organism Identification"]["Mycobacterium Reads"]
    processed_runs.at[uid,'UNCLASSIFIED_READS'] = data["Organism Identification"]["Unclassified Reads"]
    processed_runs.at[uid,'NON_MYCOBACTERIAL_READS'] = data["Organism Identification"]["Non-Mycobacterium Bacteria Reads"]
    processed_runs.at[uid,'PIPELINE_BUILD'] = data["Metadata"]["Pipeline build"].replace("\n", "")

    # now let's work out how many species have been detected
    n_species = len(data['Mycobacterium Results']['Summary'])
    for i_species in range(n_species):
        row = [uid]
        gpas_species_name = data['Mycobacterium Results']['Summary'][i_species]["Name"]

        if 'lineage' in gpas_species_name:
            if 'mixed' in gpas_species_name:
                lineage = 'Mixed'
                sublineage = None
            else:
                sublineage = gpas_species_name.split('lineage ')[1][:-1]
                lineage = sublineage[0]
                sublineage = 'L'+sublineage
                lineage = 'Lineage '+lineage
            species_name = gpas_species_name.split(' (')[0] 
            if species_name == 'M. tuberculosis':
                species_name = "Mycobacterium tuberculosis"       
        else:
            species_name = gpas_species_name
            lineage = data['Mycobacterium Results']['Lineage'][i_species]["Name"].replace('_',' ')
            sublineage = None
        
        antibiogram = ""
            
        row.append(species_name)
        row.append(lineage)
        row.append(sublineage)
        row.append(data['Mycobacterium Results']['Summary'][i_species]["Num Reads"])
        row.append(data['Mycobacterium Results']['Summary'][i_species]["Coverage"])
        row.append(data['Mycobacterium Results']['Summary'][i_species]["Depth"])
        if data["Genomes"] is not None:
            amr_results = data["Genomes"][0]["Resistance Prediction"][
                "Resistance Prediction Summary"
            ]
            for j in amr_results:
                for k in amr_results[j]:
                    antibiogram += amr_results[j][k]
            row.append(antibiogram)
        else:
            row.append("")
        tables.append(row)

foo = pandas.DataFrame(tables, columns = ['RUN_ACCESSION', 'SPECIES', 'LINEAGE', "SUBLINEAGE", 'N_READS', 'COVERAGE', 'DEPTH', 'ANTIBIOGRAM'])
foo

190eb979-d779-461a-baf9-c76d28518cce not in the master table
e3f3b423-ab5c-46d6-80de-00386845370b not in the master table
ee6bce21-e44a-4daf-ad5f-9f9aec508162 not in the master table
864a187a-7b26-46ec-9c5d-5fbb0cddedd2 not in the master table
96e915b9-23a1-4ac0-a645-c82bf6f54677 not in the master table
2fda82bc-1c2f-4142-832f-b4b6248b6fe4 not in the master table
6e388a12-6647-425f-8fd7-b3da020e1aa6 not in the master table


,RUN_ACCESSION,SPECIES,LINEAGE,SUBLINEAGE,N_READS,COVERAGE,DEPTH,ANTIBIOGRAM
0,SRR6982341,Mycobacterium tuberculosis,Lineage 2,L2.2,1353179,98.9239,61.9982,RSSSSSSSSSSSRS
1,f15c7db0-e04e-412c-a3f3-f8e2228a05e3,Mycobacterium gwanakae,Mycobacterium chelonae subsp. gwanakae,None,2862784,83.5037,54.7106,
2,37b648dd-ab0d-450c-8346-dcd0199d54a9,Mycobacterium avium,Mycobacterium avium subsp. hominissuis,None,498228,92.4300,12.2538,
3,SRR6982507,Mycobacterium tuberculosis,Lineage 4,L4.4.1.1,686139,98.6238,25.5012,SUSSUUSSSSSUSS
4,SRR6982241,Mycobacterium tuberculosis,Lineage 4,L4.4.1.1,777002,99.1051,35.4851,SSSSUUSSSSSSSS
...,...,...,...,...,...,...,...,...
285,SRR6982346,Mycobacterium tuberculosis,Lineage 2,L2.2.4,815590,99.0125,33.9705,SSSSSSSSUUSSSS
286,SRR6982464,Mycobacterium tuberculosis,Lineage 2,L2.2.7,679454,98.8376,23.6499,SSSSSSSSSSSSSS
287,c6fc8f33-0c8b-4f41-a525-88145fbcf923,Mycobacterium intracellulare,Mycobacterium intracellulare subsp. chimaera,None,1231635,87.5398,29.4110,
288,1bc5a4ab-26aa-4209-860e-cf7c4fe8717a,Mycobacterium intracellulare,Mycobacterium intracellulare subsp. yongonense,None,1221274,87.4114,26.1086,


In [134]:
processed_runs[processed_runs.has_main_report]

,RUN_ACCESSION,UNIQUEID,SAMPLE_ACCESSION,MYCOBACTERIAL_READS,UNCLASSIFIED_READS,NON_MYCOBACTERIAL_READS,PIPELINE_BUILD,in_mapping_file,has_main_report,has_mutations,has_effects,has_predictions,status,batch_name,pipeline_version,sample_name,remote_sample_name,remote_batch_name,remote_batch_id
bucket_sample_name,,,,,,,,,,,,,,,,,,,
2eccd9d5-1244-4ccb-8185-a2818c1c3934,07cd3091-1e2b-436d-b85d-c555d5053ec1,None,None,4236146,153460,0,5360392,True,True,False,False,False,too few H37Rv reads,trial4-mycobacterium-avium_batch_42,2.4.1,2eccd9d5-1244-4ccb-8185-a2818c1c3934_mycobacte...,e9fb0ed4-9913-4909-85f5-4b3f7a017c8d,0cwucm,8e3c060b-9d43-48da-a9a0-ddd16eba6044
5bdb391f-7d89-4f1e-8650-70bc0bb2a884,1046d78c-5695-4a84-a097-b01e9acd24b5,None,None,3287462,13338,0,5360392,True,True,False,False,False,too few H37Rv reads,trial4-mycobacterium-avium_batch_42,2.4.1,5bdb391f-7d89-4f1e-8650-70bc0bb2a884_mycobacte...,51f50623-5b41-4d27-a7ea-715420c6fd70,0cwucm,8e3c060b-9d43-48da-a9a0-ddd16eba6044
39afd5a7-ae77-421c-b3ec-760dd3c8e95f,1ac3695f-e438-4472-816f-5c2f372a048a,None,None,6218970,118500,0,5360392,True,True,False,False,False,too few H37Rv reads,trial4-mycobacterium-intracellulare_batch_10,2.4.1,39afd5a7-ae77-421c-b3ec-760dd3c8e95f_mycobacte...,ae6e67dc-9ba4-4ffb-b3cc-e8489a7278aa,qmapzc,f78c0fb0-3dac-45c4-921c-573157b0d72a
bd821678-1865-48bc-a836-8d97a7e99b3d,41932e11-5d1b-4b92-b78d-8d771f1885bd,None,None,2564882,17466,0,68d0e6f,True,True,False,False,False,too few H37Rv reads,trial4-mycobacterium-abscessus_batch_51,2.4.1,bd821678-1865-48bc-a836-8d97a7e99b3d_mycobacte...,e2375fb6-a3fe-49bc-8b17-6cdb0d8e9ecb,0pqwzx,00e3f459-d6a8-4f6d-a9ba-abd210f6b8d5
14adb7de-93ac-424d-9db5-ac11a74f6847,59844e32-b620-48a2-ad90-770aeb011bd2,None,None,6053360,5018,0,5360392,True,True,False,False,False,too few H37Rv reads,trial4-mycobacterium-intracellulare_batch_10,2.4.1,14adb7de-93ac-424d-9db5-ac11a74f6847_mycobacte...,423b8653-0a6f-41a8-9982-9a38a7c7f5df,qmapzc,f78c0fb0-3dac-45c4-921c-573157b0d72a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR6982036,SRR6982036,None,None,736782,22,478,5ae3bc4,True,True,False,False,False,complete,NaN,2.4.1,SRR6982036,90406ae5-4576-46b1-8499-e42840192172,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129
SRR6982033,SRR6982033,None,None,723072,1280,160636,5ae3bc4,True,True,False,False,False,complete,NaN,2.4.1,SRR6982033,43321ebd-cb71-4f3a-b7e3-84ba43d02b17,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129
SRR6982034,SRR6982034,None,None,467286,158,868,5ae3bc4,True,True,False,False,False,complete,NaN,2.4.1,SRR6982034,cf8b2308-0407-4086-8f7a-6e9780807653,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129


In [12]:
SPECIES = pandas.read_csv(tables_path / "SPECIES.csv")
SPECIES

,RUN_ACCESSION,SPECIES,SUBSPECIES,LINEAGE,SUBLINEAGE,N_READS,COVERAGE,DEPTH
0,SRR6982341,Mycobacterium tuberculosis,Mycobacterium tuberculosis,lineage2,lineage2.2,1353179,98.9239,61.9982
1,f15c7db0-e04e-412c-a3f3-f8e2228a05e3,Mycobacterium gwanakae,Mycobacterium chelonae subsp. gwanakae,NaN,NaN,2862784,83.5037,54.7106
2,37b648dd-ab0d-450c-8346-dcd0199d54a9,Mycobacterium avium,Mycobacterium avium subsp. hominissuis,NaN,NaN,498228,92.4300,12.2538
3,SRR6982507,Mycobacterium tuberculosis,Mycobacterium tuberculosis,lineage4,lineage4.4.1.1,686139,98.6238,25.5012
4,SRR6982241,Mycobacterium tuberculosis,Mycobacterium tuberculosis,lineage4,lineage4.4.1.1,777002,99.1051,35.4851
...,...,...,...,...,...,...,...,...
294,SRR6982346,Mycobacterium tuberculosis,Mycobacterium tuberculosis,lineage2,lineage2.2.4,815590,99.0125,33.9705
295,SRR6982464,Mycobacterium tuberculosis,Mycobacterium tuberculosis,lineage2,lineage2.2.7,679454,98.8376,23.6499
296,c6fc8f33-0c8b-4f41-a525-88145fbcf923,Mycobacterium intracellulare,Mycobacterium intracellulare subsp. chimaera,NaN,NaN,1231635,87.5398,29.4110
297,1bc5a4ab-26aa-4209-860e-cf7c4fe8717a,Mycobacterium intracellulare,Mycobacterium intracellulare subsp. yongonense,NaN,NaN,1221274,87.4114,26.1086


In [13]:
SPECIES.SPECIES.value_counts()

SPECIES
Mycobacterium tuberculosis      100
Mycobacterium intracellulare     51
Mycobacterium avium              50
Mycobacterium gwanakae           49
Mycobacterium abscessus          49
Name: count, dtype: int64

In [61]:
SPECIES[SPECIES.SPECIES == 'Mycobacterium tuberculosis complex'].LINEAGE.value_counts(dropna=False)

LINEAGE
lineage4    69
lineage2    25
lineage3     5
lineage1     1
Name: count, dtype: int64

In [62]:

SPECIES.LINEAGE.value_counts(dropna=False)

LINEAGE
NaN         199
lineage4     69
lineage2     25
lineage3      5
lineage1      1
Name: count, dtype: int64

In [23]:
SPECIES.SUBLINEAGE.value_counts()

SUBLINEAGE
lineage4.3.2.1      17
lineage4.10         13
lineage4.1.1.3      10
lineage2.2.4         9
lineage4.4.1.1       9
lineage2.2.7         8
lineage2.2           7
lineage4.1.2.1       5
lineage4.3.3         4
lineage4.3.4.2       4
lineage4.3.4.2.1     4
lineage3.1.1         3
lineage3             2
lineage1.1.2         1
lineage2             1
lineage4.3.4.1       1
lineage4.1.1.1       1
lineage4.1.1.2       1
Name: count, dtype: int64

In [8]:
MASTER_TABLE = pandas.read_csv(tables_path / "TEST_PROCESSED_RUNS.csv")
MASTER_TABLE

,RUN_ACCESSION,batch_name,GPAS_PIPELINE_VERSION,remote_sample_name,remote_batch_name,remote_batch_id,has_main_report,status,HUMAN_READS,MYCOBACTERIAL_READS,OTHER_BACTERIAL_READS,UNCLASSIFIED_READS,GPAS_PIPELINE_BUILD,has_new_block_in_main_report,number_of_species
0,77434f41-625f-424a-97c5-8d318ce11410,mixture-test,2.4.1,77434f41-625f-424a-97c5-8d318ce11410,8oruua,48f94831-7896-4127-bfd1-2ce5c5a034e9,True,complete,NaN,2399958.0,0.0,51662.0,fa04ac9,True,2.0
1,05c9ed42-e452-48d9-bee8-993320c428e8,trial4-mycobacterium-avium_batch_42,2.4.1,562ab3e6-b0a7-4b5c-8545-4e287257e4c8,0cwucm,8e3c060b-9d43-48da-a9a0-ddd16eba6044,True,cannot assemble,NaN,1194206.0,0.0,85508.0,5360392,False,1.0
2,05f52b1f-5762-4b0a-9f57-2c5c268b5472,trial4-mycobacterium-avium_batch_42,2.4.1,47bb4d52-e76b-4ee6-96f1-2391b15983a8,0cwucm,8e3c060b-9d43-48da-a9a0-ddd16eba6044,False,NaN,NaN,NaN,NaN,NaN,NaN,False,0.0
3,0aed009d-55b8-4fe3-b1c3-f1ed3c86fcbb,trial4-mycobacterium-avium_batch_42,2.4.1,7d035e33-60ab-410c-8874-b6aa056096b7,0cwucm,8e3c060b-9d43-48da-a9a0-ddd16eba6044,True,cannot assemble,NaN,1948244.0,0.0,37806.0,5360392,False,1.0
4,10f45b3e-bc83-4576-b5d4-10b47075390a,trial4-mycobacterium-avium_batch_42,2.4.1,8600feb4-616d-4ffe-b79f-2adda7264be4,0cwucm,8e3c060b-9d43-48da-a9a0-ddd16eba6044,True,cannot assemble,NaN,4707150.0,0.0,83992.0,5360392,False,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,SRR6982036,TB test batch,2.4.1,90406ae5-4576-46b1-8499-e42840192172,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129,True,complete,NaN,736782.0,478.0,22.0,5ae3bc4,False,1.0
297,SRR6982033,TB test batch,2.4.1,43321ebd-cb71-4f3a-b7e3-84ba43d02b17,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129,True,complete,NaN,723072.0,160636.0,1280.0,5ae3bc4,False,1.0
298,SRR6982034,TB test batch,2.4.1,cf8b2308-0407-4086-8f7a-6e9780807653,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129,True,complete,NaN,467286.0,868.0,158.0,5ae3bc4,False,1.0
299,SRR6982446,TB test batch,2.4.1,f02f6ce1-c6ba-4c74-9a44-37c635298b1e,0f0zao,c5bfc0fe-c5ad-48c2-836e-e430d42cf129,True,complete,NaN,623266.0,576.0,10.0,5ae3bc4,False,1.0


In [9]:
MASTER_TABLE[MASTER_TABLE.has_new_block_in_main_report]

,RUN_ACCESSION,batch_name,GPAS_PIPELINE_VERSION,remote_sample_name,remote_batch_name,remote_batch_id,has_main_report,status,HUMAN_READS,MYCOBACTERIAL_READS,OTHER_BACTERIAL_READS,UNCLASSIFIED_READS,GPAS_PIPELINE_BUILD,has_new_block_in_main_report,number_of_species
0,77434f41-625f-424a-97c5-8d318ce11410,mixture-test,2.4.1,77434f41-625f-424a-97c5-8d318ce11410,8oruua,48f94831-7896-4127-bfd1-2ce5c5a034e9,True,complete,NaN,2399958.0,0.0,51662.0,fa04ac9,True,2.0


In [11]:
SPECIES[SPECIES.RUN_ACCESSION=='77434f41-625f-424a-97c5-8d318ce11410']

,RUN_ACCESSION,SPECIES,SUBSPECIES,LINEAGE,SUBLINEAGE,N_READS,COVERAGE,DEPTH
231,77434f41-625f-424a-97c5-8d318ce11410,Mycobacterium intracellulare complex,Mycobacterium intracellulare subsp. intracellu...,NaN,NaN,1806529,93.0141,45.40260
232,77434f41-625f-424a-97c5-8d318ce11410,Mycobacterium avium,NaN,NaN,NaN,217986,90.6699,5.93603
